# ✈️  Mapeando colunas diferentes com as do Banco de Dados | Enviando para o Postgree

Versão limpa e refatorada do pipeline de dados, organizada em 4 passos lógicos:
- **Passo 1:** Importação das bibliotecas (`pandas` e `psycopg2`)
- **Passo 2:** Extração dos dados (leitura do arquivo JSON)
- **Passo 3:** Transformação dos dados (filtragem e remoção de acentos)
- **Passo 4:** Carga no PostgreSQL (limpeza com TRUNCATE, inserção em lote e validação)


In [1]:
# ==============================================================================
# PASSO 1: IMPORTAÇÃO DAS BIBLIOTECAS
# ==============================================================================

import pandas as pd     # Manipulação e estruturação de tabelas (DataFrames)
import psycopg2         # Driver conector com o banco de dados PostgreSQL


In [2]:
# ==============================================================================
# PASSO 2: EXTRAÇÃO DOS DADOS (LEITURA DO JSON)
# ==============================================================================

# Caminho do arquivo JSON (o prefixo 'r' define raw string e evita erros de barras)
caminho_do_arquivo = r"/Users/analeia/Desktop/Engenharia de Dados/V_OCORRENCIA_AMPLA.json"

# Lê o JSON e carrega na memória como tabela. utf-8-sig preserva acentos corretamente
db = pd.read_json(caminho_do_arquivo, encoding="utf-8-sig")

db.head()  # Exibe as primeiras linhas do DataFrame para conferência


,Numero_da_Ocorrencia,Numero_da_Ficha,Operador_Padronizado,Classificacao_da_Ocorrência,Data_da_Ocorrencia,Hora_da_Ocorrência,Municipio,UF,Regiao,Descricao_do_Tipo,...,Lesoes_Desconhecidas_Tripulantes,Lesoes_Desconhecidas_Passageiros,Lesoes_Desconhecidas_Terceiros,Modelo,CLS,Tipo_ICAO,PMD,Numero_de_Assentos,Nome_do_Fabricante,PSSO
0,33947,2022722007,AZUL LINHAS AÉREAS BRASILEIRAS S.A.,Incidente,2022-08-11,05:31:00,PALMAS,TO,Norte,FALHA OU MAU FUNCIONAMENTO DE SISTEMA / COMPON...,...,NaN,NaN,NaN,ERJ 190-200 LR,L2J,E195,50790.0,125.0,EMBRAER,verdadeiro
1,29123,202140534,AZUL,Incidente Grave,2021-08-06,16:59,CAXIAS DO SUL,RS,Sul,CONTATO ANORMAL COM A PISTA,...,NaN,NaN,NaN,ERJ 190-400,L2J,E295,61500.0,143.0,EMBRAER,verdadeiro
2,32189,2022293670,FIGUEIREDO AVIAÇÃO AGRÍCOLA EIRELI,Incidente Grave,2022-05-21,12:25,DOURADOS,MS,Centro-Oeste,OPERAÇÃO A BAIXA ALTITUDE,...,NaN,NaN,NaN,EMB-201,L1P,IPAN,1800.0,1.0,NEIVA,verdadeiro
3,32190,2022354545,FENIX PARTICIPACOES LTDA,Incidente,2022-02-12,15:30,CAMPINAS,SP,Sudeste,COLISÃO COM AVE,...,NaN,NaN,NaN,PC-12/47E,L1T,PC12,4740.0,10.0,PILATUS,verdadeiro
4,32726,2022125796,ANAPE EMPREENDIMENTOS IMOBILIÁRIOS-EIRELI,Incidente,2022-06-26,13:45,CAMPINAS,SP,Sudeste,COLISÃO NO SOLO,...,NaN,NaN,NaN,A109E,H2T,A109,3000.0,8.0,AGUSTA,verdadeiro


In [3]:
# ==============================================================================
# PASSO 3: TRANSFORMAÇÃO (FILTRAR E PADRONIZAR NOMES DE COLUNAS)
# ==============================================================================

# 1. Lista apenas as 7 colunas necessárias para o banco de dados
colunas = [
    "Numero_da_Ocorrencia",         # Identificador único da ocorrência
    "Classificacao_da_Ocorrência",  # Gravidade (ainda com acento no arquivo original)
    "Data_da_Ocorrencia",           # Data do evento
    "Municipio",                    # Município/cidade
    "UF",                           # Estado
    "Regiao",                       # Região do Brasil
    "Nome_do_Fabricante",           # Fabricante da aeronave
    "Modelo"          
]

# Proteção contra reexecução: se a célula for executada novamente, evita KeyError
if 'Classificacao_da_Ocorrência' not in db.columns and 'Classificacao_da_Ocorrencia' in db.columns:
    colunas[1] = 'Classificacao_da_Ocorrencia'

# Filtra o DataFrame mantendo apenas as colunas da lista
db = db[colunas]

# 2. Remove o acento da coluna (boa prática em banco de dados relacional)
if 'Classificacao_da_Ocorrência' in db.columns:
    db = db.rename(columns={'Classificacao_da_Ocorrência': 'Classificacao_da_Ocorrencia'})

db.head(3)

,Numero_da_Ocorrencia,Classificacao_da_Ocorrencia,Data_da_Ocorrencia,Municipio,UF,Regiao,Nome_do_Fabricante,Modelo
0,33947,Incidente,2022-08-11,PALMAS,TO,Norte,EMBRAER,ERJ 190-200 LR
1,29123,Incidente Grave,2021-08-06,CAXIAS DO SUL,RS,Sul,EMBRAER,ERJ 190-400
2,32189,Incidente Grave,2022-05-21,DOURADOS,MS,Centro-Oeste,NEIVA,EMB-201


In [4]:
# ==============================================================================
# PASSO 4: CARGA NO POSTGRESQL E VALIDAÇÃO
# ==============================================================================

# 1. Parâmetros de acesso ao banco PostgreSQL
dbname = 'python'
user = 'analeia'
password = '12345'
host = 'localhost'
port = '5432'

# 2. Abre a conexão com o banco e cria o cursor (executor de comandos SQL)
conn = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
cur = conn.cursor()

# 3. Cria a tabela anac_mapeamento no banco caso ela ainda não exista (com a coluna Modelo)
cur.execute("""
CREATE TABLE IF NOT EXISTS anac_mapeamento (
    ID INT,
    Classificacao_da_Ocorrencia VARCHAR(50),
    Dt_da_Ocorrencia DATE,
    Municipio VARCHAR(50),
    UF VARCHAR(30),
    Regiao VARCHAR(30),
    Fabri VARCHAR(100),
    Modelo VARCHAR(50)
);
""")

# Garante que a coluna Modelo exista caso a tabela tenha sido criada anteriormente sem ela
cur.execute("ALTER TABLE anac_mapeamento ADD COLUMN IF NOT EXISTS Modelo VARCHAR(50);")

# 4. Limpa dados antigos da tabela para não duplicar registros a cada execução
cur.execute("TRUNCATE TABLE anac_mapeamento;")

# 5. Percorre cada linha do DataFrame e insere na tabela do banco (fazendo o De-Para)
for indice, coluna in db.iterrows():
    cur.execute("""
    INSERT INTO anac_mapeamento (
        ID,
        Classificacao_da_Ocorrencia,
        Dt_da_Ocorrencia,
        Municipio,
        UF,
        Regiao,
        Fabri,
        Modelo
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        coluna['Numero_da_Ocorrencia'],        # Mapeia para ID
        coluna['Classificacao_da_Ocorrencia'],  # Mapeia para Classificacao_da_Ocorrencia
        coluna['Data_da_Ocorrencia'],           # Mapeia para Dt_da_Ocorrencia
        coluna['Municipio'],                    # Mapeia para Municipio
        coluna['UF'],                           # Mapeia para UF
        coluna['Regiao'],                       # Mapeia para Regiao
        coluna['Nome_do_Fabricante'],            # Mapeia para Fabri
        coluna['Modelo']                        # Mapeia para Modelo
    ))

# 6. Grava permanentemente todas as alterações no banco de dados
conn.commit()

# 7. Validação: confere se a contagem de linhas no banco bate com o DataFrame
cur.execute("SELECT COUNT(*) FROM anac_mapeamento;")
total_banco = cur.fetchone()[0]

print("-" * 45)
print(f"Linhas no DataFrame: {len(db)}")
print(f"Linhas no Banco:     {total_banco}")
if len(db) == total_banco:
    print("🎉 Sucesso! Carga finalizada com 100% de integridade na tabela anac_mapeamento.")
else:
    print("⚠️ Atenção: A quantidade de linhas divergiu.")
print("-" * 45)

# 8. Fecha o cursor e a conexão para liberar os recursos do servidor
cur.close()
conn.close()


---------------------------------------------
Linhas no DataFrame: 12221
Linhas no Banco:     12221
🎉 Sucesso! Carga finalizada com 100% de integridade na tabela anac_mapeamento.
---------------------------------------------
